# Step 19 — what does federation actually buy?

Steps 16 and 18 showed that a module definition can be carried into a cohort that never discovered
it, and that the pooling is exact to ~1e-12. Neither asked the question that matters: **does it
help?**

This step measures it, for every eligible module scored in every cohort, at three levels:

| level | what the cohort has | what leaves a site |
|---|---|---|
| **alone** | only its own WGCNA fit — if the module is not in it, nothing | nothing |
| **+ shared definition** | the origin's **protein list**; loadings and scaling from its own data | a list of protein names |
| **+ federated** | loadings from PC1 of the **pooled** correlation, pooled centring and scaling | `N, S, Q, G` for that module |

The levels are cumulative and the difference between them is the thing being priced. Level 2 is
projection and releases no Gram at all; level 3 is federation proper.

Reads `wgcna_{A,B,C}.rds` and `step12_panels.rds`. Writes `step19_benefit.rds` and two figures.

In [ ]:
options(stringsAsFactors=FALSE); set.seed(42)
source("../src/paths.R")
SITES <- c("A","B","C"); FDR <- 0.05
W <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES
P <- readRDS(art("step12_panels.rds")); D <- P$D; spec <- P$spec
N <- sapply(W, function(w) nrow(w$X)); N_MIN <- min(N)

build_traits <- function(m, spec){ out<-data.frame(row.names=rownames(m))
  for (i in seq_len(nrow(spec))){ v<-m[[spec$source_column[i]]]
    out[[spec$name[i]]] <- switch(spec$type[i], numeric=as.numeric(as.character(v)),
      binary=as.numeric(v=="Positive"),
      ordinal={lvl<-unique(v[!is.na(v)&v!=""]); lvl<-lvl[order(as.numeric(sub("-.*","",lvl)))]
               as.integer(factor(v,levels=lvl,ordered=TRUE))}) }; out }
TR <- lapply(SITES, function(s) build_traits(W[[s]]$meta, spec)); names(TR) <- SITES

suff <- function(M){ M<-as.matrix(M)
  list(N=crossprod(!is.na(M)), S=crossprod(!is.na(M),replace(M,is.na(M),0)),
       Q=crossprod(!is.na(M),replace(M,is.na(M),0)^2), G=crossprod(replace(M,is.na(M),0))) }
pooled_cor <- function(st){ n<-st$N; s<-st$S; g<-st$G; q<-st$Q
  cv <- g/n-(s*t(s))/(n*n); sv <- sqrt(q/n-(s/n)^2); pmin(pmax(cv/(sv*t(sv)),-1),1) }
pc1 <- function(R){ v<-eigen(R,symmetric=TRUE)$vectors[,1]; if (sum(v)<0) v<--v; v }

# trait association vector for one score
assoc <- function(sc, s){
  tr <- TR[[s]][names(sc), , drop=FALSE]
  r  <- cor(sc, tr, use="pairwise.complete.obs")[1,]
  p  <- 2*pt(-abs(r*sqrt((N[s]-2)/(1-r^2))), N[s]-2)
  list(r=r, q=p.adjust(p,"BH"))
}

ELIG <- do.call(rbind, lapply(SITES, function(o){
  d <- D[[paste(o,"all15")]]; if (!length(d$keep)) return(NULL)
  do.call(rbind, lapply(d$keep, function(k){
    g <- colnames(W[[o]]$X)[W[[o]]$mods==k]
    if (n_proteins(g) >= N_MIN) return(NULL)
    data.frame(origin=o, module=k, nprot=n_proteins(g)) })) }))
cat(sprintf("eligible modules: %d (A %d, C %d)\n\n", nrow(ELIG),
            sum(ELIG$origin=="A"), sum(ELIG$origin=="C")))

LEV <- c("alone","shared definition","federated")
rows <- list(); dr <- list()
for (i in seq_len(nrow(ELIG))){
  o <- ELIG$origin[i]; k <- ELIG$module[i]
  g <- colnames(W[[o]]$X)[W[[o]]$mods==k]
  st  <- Reduce(function(a,b) Map(`+`,a,b), lapply(SITES, function(s) suff(W[[s]]$X[,g,drop=FALSE])))
  Rf  <- pooled_cor(st); vf <- pc1(Rf)
  nt  <- st$N[1,1]; muf <- st$S[1,]/nt; sdf <- sqrt(st$Q[1,]/nt - muf^2)
  for (s in SITES){
    X  <- W[[s]]$X[, g, drop=FALSE]
    s2 <- setNames(as.vector(scale(X) %*% pc1(cor(X))), rownames(X))            # local loadings
    s3 <- setNames(as.vector(scale(X, center=muf, scale=sdf) %*% vf), rownames(X))
    a2 <- assoc(s2, s); a3 <- assoc(s3, s)
    # level 1: the module has to exist in this cohort's OWN fit
    n1 <- if (s == o) sum(a2$q < FDR) else 0L
    rows[[length(rows)+1]] <- data.frame(origin=o, module=k, nprot=ELIG$nprot[i], site=s,
      alone=n1, shared=sum(a2$q<FDR), federated=sum(a3$q<FDR),
      score_cor=round(abs(cor(s2,s3)),3), own_fit=(s==o))
    dr[[length(dr)+1]] <- data.frame(site=s, module=k, trait=names(a2$r),
      d_absr = abs(a3$r) - abs(a2$r))
  }
}
res <- do.call(rbind, rows); DR <- do.call(rbind, dr)

cat("per cohort, associations at FDR 5%:\n")
agg <- aggregate(cbind(alone,shared,federated) ~ site, res, sum)
print(agg, row.names=FALSE)
cat(sprintf("\nTOTAL  alone %d  ->  shared definition %d  ->  federated %d\n",
            sum(res$alone), sum(res$shared), sum(res$federated)))
cat(sprintf("score correlation, shared vs federated: %.3f - %.3f\n",
            min(res$score_cor), max(res$score_cor)))
cat(sprintf("\nDelta |r| (federated - shared) over %d cohort x module x trait cells:\n", nrow(DR)))
print(round(summary(DR$d_absr), 4))
cat(sprintf("  |delta| > 0.05 in %d of %d cells (%.1f%%)\n",
            sum(abs(DR$d_absr)>0.05), nrow(DR), 100*mean(abs(DR$d_absr)>0.05)))
cat("\nmodules where federation changed the count:\n")
ch <- res[res$federated != res$shared, c("origin","module","site","shared","federated","score_cor")]
print(ch, row.names=FALSE)
saveRDS(list(res=res, delta=DR, elig=ELIG, levels=LEV), art("step19_benefit.rds"))


## The headline, and a trap in it

**Level 1 → 2 is where everything happens.** 38 → 81 associations. Cohort B goes from **0 to 11**:
B's own fit contains no module resembling these at all, so without a shared protein list it has
nothing to score.

**Level 2 → 3 looks like a small gain: 81 → 85.** It is not a gain. Two things give it away — the
level-2 vs level-3 score correlations run **0.968 to 1.000**, and the change in effect size is a
distribution centred on zero, with `|Δ|r|| > 0.05` in **2 of 540** cells.

The entire +4 comes from two modules, and the cell below shows what is really happening in the one
that moved.

In [ ]:
# Why mediumpurple3 at B "gains" five associations while its score is unchanged.
g  <- colnames(W$C$X)[W$C$mods == "mediumpurple3"]
st <- Reduce(function(a,b) Map(`+`,a,b), lapply(SITES, function(s) suff(W[[s]]$X[,g,drop=FALSE])))
Rf <- pooled_cor(st); vf <- pc1(Rf)
nt <- st$N[1,1]; muf <- st$S[1,]/nt; sdf <- sqrt(st$Q[1,]/nt - muf^2)
X  <- W$B$X[, g, drop = FALSE]
s2 <- setNames(as.vector(scale(X) %*% pc1(cor(X))), rownames(X))
s3 <- setNames(as.vector(scale(X, center = muf, scale = sdf) %*% vf), rownames(X))
a2 <- assoc(s2, "B"); a3 <- assoc(s3, "B")
cmp <- data.frame(trait = names(a2$r),
                  r_shared = round(a2$r, 3), q_shared = round(a2$q, 4),
                  r_fed    = round(a3$r, 3), q_fed    = round(a3$q, 4),
                  crossed  = (a2$q >= FDR) & (a3$q < FDR))
cat(sprintf("score correlation, shared vs federated: %.6f\n\n", cor(s2, s3)))
print(cmp[order(cmp$q_fed), ], row.names = FALSE)

## The +5 is a Benjamini–Hochberg boundary effect, not a finding

Read the rows that crossed:

- `SLEDAI_2K` — r **0.235 → 0.235**, identical to three decimals, q 0.0516 → 0.0472
- `C4_level` — r **−0.237 → −0.237**, identical, q 0.0516 → 0.0472
- `RNP_A_status` — r 0.240 → 0.237, the effect size **got smaller**, and it crossed anyway

The score correlation is **0.9998**. Nine traits are piled between q = 0.046 and q = 0.052, and BH
is a rank-based staircase over the whole set: a negligible shift in one p-value reorders the ranks
and tips five traits across the line together.

**Counting associations at a threshold is the wrong metric when the p-values are stacked at the
threshold.** The effect-size distribution in the second figure is the honest view, and it sits on
zero.

## The figures

Bars for the headline; slopes beneath so the summary cannot be read without the spread behind it.
A third figure shows the Δ|r| distribution directly.

In [ ]:
# ---- figures: bars for the headline, slopes for the spread ---------------
agg <- aggregate(cbind(alone, shared, federated) ~ site, res, sum)
LEVCOL <- c(alone = "#BDBDBD", shared = "#2166AC", federated = "#B2182B")

png(art("fig19_federation_benefit.png"), width = 2600, height = 2000, res = 170)
layout(matrix(c(1,2), nrow = 2), heights = c(1, 1.25))

par(mar = c(4, 5, 4, 2))
M <- t(as.matrix(agg[, c("alone","shared","federated")])); colnames(M) <- agg$site
bp <- barplot(M, beside = TRUE, col = LEVCOL, border = NA, ylim = c(0, max(M) * 1.45),
              ylab = "trait associations at FDR 5%", xlab = "cohort",
              main = "What each cohort gains, and from what")
text(bp, M + max(M)*0.04, M, cex = 0.85)
legend("topleft", bty = "n", cex = 0.95, fill = LEVCOL, border = NA,
       legend = c("alone: this cohort's own discovery",
                  "+ shared protein list (projection)",
                  "+ federated loadings (pooled N,S,Q,G)"))
mtext("12 eligible modules (9 from A, 3 from C), each scored in every cohort",
      side = 3, line = 0.2, cex = 0.75)

par(mar = c(4, 5, 4, 9))
ymax <- max(res$federated, res$shared, res$alone)
plot(NA, xlim = c(0.85, 3.15), ylim = c(-0.4, ymax + 0.4), xaxt = "n",
     xlab = "", ylab = "associations at FDR 5%",
     main = "Every cohort x module separately -- 36 lines")
axis(1, at = 1:3, labels = c("alone", "shared definition", "federated"))
SITECOL <- c(A = "#1B7837", B = "#762A83", C = "#E08214")
set.seed(42)
for (i in seq_len(nrow(res))) {
  j <- runif(1, -0.06, 0.06)
  y <- c(res$alone[i], res$shared[i], res$federated[i]) + j
  lines(1:3, y, col = adjustcolor(SITECOL[res$site[i]], 0.55), lwd = 1.6)
  points(1:3, y, col = SITECOL[res$site[i]], pch = 16, cex = 0.6)
}
ch <- which(res$federated != res$shared)
for (i in ch) {
  text(3.05, res$federated[i], sprintf("%s @ %s", res$module[i], res$site[i]),
       adj = 0, cex = 0.7, col = SITECOL[res$site[i]], xpd = TRUE)
}
legend("topleft", bty = "n", cex = 0.85, lwd = 2, col = SITECOL,
       legend = paste("cohort", names(SITECOL)))
mtext("nearly every line is flat between the last two levels", side = 3, line = 0.2, cex = 0.75)
invisible(dev.off())
cat("wrote", basename(art("fig19_federation_benefit.png")), "\n")

png(art("fig19_delta_effectsize.png"), width = 2000, height = 1100, res = 170)
par(mar = c(4.5, 5, 4, 2))
h <- hist(DR$d_absr, breaks = 60, col = "#2166AC", border = "white",
          xlab = expression(Delta*"|r|   (federated  -  shared definition)"),
          main = "Federated loadings change effect sizes by essentially nothing")
abline(v = 0, lwd = 2, col = "#B2182B")
mtext(sprintf("%d cohort x module x trait cells | median %.4f | |delta| > 0.05 in %d (%.1f%%)",
              nrow(DR), median(DR$d_absr), sum(abs(DR$d_absr) > 0.05),
              100*mean(abs(DR$d_absr) > 0.05)), side = 3, line = 0.2, cex = 0.8)
invisible(dev.off())
cat("wrote", basename(art("fig19_delta_effectsize.png")), "\n")


## Conclusion

**Federation is worth doing here for safety and a common vocabulary, not for power.**

A cohort gains a great deal from *being told which proteins form a module* — that is the 38 → 81
step, and for cohort B it is the difference between having nothing to score and having eleven
associations. It gains **nothing measurable** from *pooling the estimate of that module's first
component*: the loadings move by less than 0.05 in |r| in 538 of 540 cells, and the apparent 81 →
85 is a multiple-testing boundary artifact.

That matters for how a federated study should be designed, because the two levels have completely
different privacy costs. **A protein list is a list of names.** A Gram matrix is the thing that has
to satisfy `p < n` and be released one module at a time. On this data the privacy-cheapest option
delivers essentially all of the benefit.

**Scope.** This is a statement about 12 modules of 7–78 proteins at n ≈ 87 per cohort. PC1 of a
small correlation matrix is already well determined at that sample size, which is exactly why
tripling n does not move it. A study with smaller cohorts, larger modules, or a target that
depends on the full covariance rather than its leading eigenvector could easily come out the other
way — and this notebook is the way to check rather than assume.